In [1]:
import torch 
import torch.nn as nn
import numpy as np 
from datasets import load_dataset
from tqdm import tqdm 
import yaml 

torch.set_float32_matmul_precision('medium')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

import importlib

import robustness.audio_functions.audio_transforms as at 
from lightning_scripts.lightning_ssl import LitAudioSSL 
import robustness.audio_models as architectures

import sys
import torchaudio 
sys.path.append('byol-a')
from byol_a.common import *
from byol_a.augmentations import PrecomputedNorm
from byol_a.models import AudioNTT2020
from easydict import EasyDict

/mnt/ceph/users/igriffith/projects/cochdnn/byol-a/byol_a/common.py:31: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("sox_io")


### Try lightning module 

In [2]:
import lightning_scripts.eval_speech_commands_transfer as sc_transfer
import lightning as L

importlib.reload(sc_transfer)
BYOLAClassifier = sc_transfer.BYOLAClassifier

config_path = 'byol-a/config.yaml'
config = load_yaml_config(config_path)

config['model'] = {}
config['hparas'] = {}
config['audio_transforms'] = {} 
# config['audio_transforms']['low_snr'] = -10
# config['audio_transforms']['high_snr'] = 10
# config['audio_transforms']['rms_level'] = 60
config['model']['arch_kwargs'] = {}
config['data'] = {}

config['num_workers'] = 2
config['num_gpus'] = 1
config['hparas']['batch_size'] = 32
config['hparas']['global_batch_size'] = 32
config['data']['eval_max'] = 3
config['hparas']['optimizer'] = "AdamW"
# used 2 gpus for training, mult by 2 for now to get same checkpoint 
config['hparas']['lr'] = 0.001
config['hparas']['epochs'] = 2
# don't load in classifier head if it exists 
config['model']['arch_kwargs']['supervised'] =  False

config['hparas']['lr_schedule'] = True
config['hparas']['num_warmup_steps_or_ratio'] = 0


byola_module = BYOLAClassifier(config)

/mnt/ceph/users/igriffith/projects/cochdnn/byol-a/byol_a/models.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = state_dict or torch.load(weight_file, map_l

In [3]:
trainer = L.Trainer(devices=1)
trainer.fit(byola_module)

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3. ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/pytorch/loops/utilities.py:72: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 2 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this Da

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [22]:
!nvidia-smi

Thu Feb 27 14:15:09 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-SXM2-32GB           Off |   00000000:AF:00.0 Off |                    0 |
| N/A   35C    P0             56W /  300W |     398MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----